# 第6章 策略控制算法

## 目录1. [环境搭建](#1-环境搭建)2. [SARSA算法](#2-SARSA算法)3. [Q-learning算法](#3-Q-learning算法)4. [悬崖漫步问题](#4-悬崖漫步问题)5. [编程题](#5-编程题)

---

## 1. 环境搭建

In [ ]:
!pip install numpy matplotlib gym pygame torch -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import random

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("环境搭建完成！")

---

## 2. SARSA算法

### 2.1 SARSA算法原理

**SARSA（State-Action-Reward-State-Action）**是在线策略TD控制的经典实现。

核心更新公式：
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left( R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t) \right)$$

SARSA是**在线策略**：更新的策略与采样的策略相同，考虑探索风险。

In [ ]:
class CliffWalkingEnv:
    """悬崖漫步环境"""
    def __init__(self):
        self.rows = 4
        self.cols = 12
        self.start = (3, 0)
        self.goal = (3, 11)
        self.cliff = [(3, i) for i in range(1, 11)]
        self.state = self.start
    
    def reset(self):
        self.state = self.start
        return self.state
    
    def step(self, action):
        dr, dc = [(-1, 0), (1, 0), (0, -1), (0, 1)][action]
        new_row = max(0, min(self.rows - 1, self.state[0] + dr))
        new_col = max(0, min(self.cols - 1, self.state[1] + dc))
        new_state = (new_row, new_col)
        
        if new_state in self.cliff:
            reward = -100
            done = True
        elif new_state == self.goal:
            reward = 0
            done = True
        else:
            reward = -1
            done = False
        
        self.state = new_state
        return new_state, reward, done

    def get_state_index(self, state):
        return state[0] * self.cols + state[1]

# 测试环境
print("测试悬崖漫步环境...")
env = CliffWalkingEnv()
state = env.reset()
print(f"起点: {state}, 目标: {env.goal}")
print(f"悬崖区域: {env.cliff[:3]}...")

In [ ]:
class SARSA:
    """SARSA算法"""
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=1.0, epsilon=0.1):
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.q_table = np.zeros((n_states, n_actions))
    
    def choose_action(self, state):
        """ε-贪心选择动作"""
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        return np.argmax(self.q_table[state])
    
    def update(self, state, action, reward, next_state, next_action):
        """SARSA更新"""
        target = reward + self.gamma * self.q_table[next_state, next_action]
        td_error = target - self.q_table[state, action]
        self.q_table[state, action] += self.alpha * td_error
    
    def get_q_table(self):
        return self.q_table.copy()

# 测试SARSA
print("测试SARSA...")
sarsa = SARSA(4*12, 4, alpha=0.1, gamma=1.0, epsilon=0.1)
env = CliffWalkingEnv()

for episode in range(100):
    state = env.reset()
    state_idx = env.get_state_index(state)
    action = sarsa.choose_action(state_idx)
    done = False
    
    while not done:
        next_state, reward, done = env.step(action)
        next_state_idx = env.get_state_index(next_state)
        next_action = sarsa.choose_action(next_state_idx)
        sarsa.update(state_idx, action, reward, next_state_idx, next_action)
        state_idx = next_state_idx
        action = next_action
    
    if episode % 20 == 0:
        print(f"  Episode {episode} 完成")

print("SARSA训练完成")

### 2.2 不同ε值对比

In [ ]:
def compare_epsilon(n_episodes=500):
    """对比不同ε值"""
    epsilons = [0.01, 0.05, 0.1, 0.2]
    colors = ['blue', 'orange', 'green', 'red']
    
    plt.figure(figsize=(12, 5))
    
    for eps, color in zip(epsilons, colors):
        rewards_list = []
        agent = SARSA(4*12, 4, alpha=0.1, gamma=1.0, epsilon=eps)
        
        for ep in range(n_episodes):
            env = CliffWalkingEnv()
            state = env.reset()
            state_idx = env.get_state_index(state)
            action = agent.choose_action(state_idx)
            total_reward = 0
            done = False
            
            while not done:
                next_state, reward, done = env.step(action)
                next_state_idx = env.get_state_index(next_state)
                next_action = agent.choose_action(next_state_idx)
                agent.update(state_idx, action, reward, next_state_idx, next_action)
                total_reward += reward
                state_idx = next_state_idx
                action = next_action
            
            rewards_list.append(total_reward)
        
        avg_rewards = np.convolve(rewards_list, np.ones(50)/50, mode='valid')
        plt.plot(avg_rewards, label=f'ε={eps}', color=color, linewidth=2)
    
    plt.xlabel('回合数', fontsize=12)
    plt.ylabel('平均奖励（50回合滑动平均）', fontsize=12)
    plt.title('SARSA: 不同ε值对比', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('sarsa_epsilon.png', dpi=150)
    plt.show()

compare_epsilon()

---

## 3. Q-learning算法

### 3.1 Q-learning原理

**Q-learning**是离线策略TD控制的经典实现。

核心更新公式：
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left( R_{t+1} + \gamma \max_a Q(S_{t+1}, a) - Q(S_t, A_t) \right)$$

Q-learning是**离线策略**：优化的策略与采样的策略解耦，更新时使用贪婪动作。

In [ ]:
class QLearning:
    """Q-learning算法"""
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=1.0, epsilon=0.1):
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.q_table = np.zeros((n_states, n_actions))
    
    def choose_action(self, state):
        """ε-贪心选择动作"""
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        return np.argmax(self.q_table[state])
    
    def update(self, state, action, reward, next_state):
        """Q-learning更新（使用贪婪动作）"""
        max_next_q = np.max(self.q_table[next_state])
        target = reward + self.gamma * max_next_q
        td_error = target - self.q_table[state, action]
        self.q_table[state, action] += self.alpha * td_error
    
    def get_q_table(self):
        return self.q_table.copy()

# 测试Q-learning
print("测试Q-learning...")
qlearn = QLearning(4*12, 4, alpha=0.1, gamma=1.0, epsilon=0.1)
env = CliffWalkingEnv()

for episode in range(100):
    state = env.reset()
    state_idx = env.get_state_index(state)
    action = qlearn.choose_action(state_idx)
    done = False
    
    while not done:
        next_state, reward, done = env.step(action)
        next_state_idx = env.get_state_index(next_state)
        qlearn.update(state_idx, action, reward, next_state_idx)
        action = qlearn.choose_action(next_state_idx)
        state_idx = next_state_idx
    
    if episode % 20 == 0:
        print(f"  Episode {episode} 完成")

print("Q-learning训练完成")

### 3.2 SARSA vs Q-learning对比

In [ ]:
def compare_sarsa_qlearning(n_episodes=500):
    """对比SARSA与Q-learning"""
    algorithms = [('SARSA', SARSA), ('Q-learning', QLearning)]
    colors = ['blue', 'red']
    
    plt.figure(figsize=(14, 5))
    
    for (name, Algo), color in zip(algorithms, colors):
        rewards_list = []
        cliff_falls = []
        
        for ep in range(n_episodes):
            agent = Algo(4*12, 4, alpha=0.1, gamma=1.0, epsilon=0.1)
            env = CliffWalkingEnv()
            state = env.reset()
            state_idx = env.get_state_index(state)
            action = agent.choose_action(state_idx)
            total_reward = 0
            done = False
            cliff_count = 0
            
            while not done:
                next_state, reward, done = env.step(action)
                next_state_idx = env.get_state_index(next_state)
                
                if reward == -100:
                    cliff_count += 1
                
                if name == 'SARSA':
                    next_action = agent.choose_action(next_state_idx)
                    agent.update(state_idx, action, reward, next_state_idx, next_action)
                else:
                    agent.update(state_idx, action, reward, next_state_idx)
                
                total_reward += reward
                state_idx = next_state_idx
                action = agent.choose_action(next_state_idx)
            
            rewards_list.append(total_reward)
            cliff_falls.append(cliff_count)
        
        avg_rewards = np.convolve(rewards_list, np.ones(50)/50, mode='valid')
        avg_cliff = np.convolve(cliff_falls, np.ones(50)/50, mode='valid')
        
        plt.subplot(1, 2, 1)
        plt.plot(avg_rewards, label=name, color=color, linewidth=2)
        plt.xlabel('回合数', fontsize=12)
        plt.ylabel('平均奖励', fontsize=12)
        plt.title('奖励曲线对比', fontsize=14)
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 2, 2)
        plt.plot(avg_cliff * 100, label=name, color=color, linewidth=2)
        plt.xlabel('回合数', fontsize=12)
        plt.ylabel('悬崖坠落率(%)', fontsize=12)
        plt.title('安全性对比', fontsize=14)
        plt.legend()
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('sarsa_vs_qlearning.png', dpi=150)
    plt.show()

compare_sarsa_qlearning()

---

## 4. 编程题

### 编程题1

**实现SARSA算法，在Windy Gridworld环境中训练智能体，观察不同ε值对学习曲线的影响。**

In [ ]:
"""
编程题1：SARSA在Windy Gridworld环境
"""
class WindyGridworldEnv:
    """Windy Gridworld环境"""
    def __init__(self):
        self.rows = 7
        self.cols = 10
        self.start = (3, 0)
        self.goal = (3, 7)
        self.wind = [0, 0, 0, 1, 1, 1, 0, 0, -1, -1]  # 每列的风力
        self.state = self.start
    
    def reset(self):
        self.state = self.start
        return self.state
    
    def step(self, action):
        dr, dc = [(-1, 0), (1, 0), (0, -1), (0, 1)][action]
        wind = self.wind[self.state[1]]
        new_row = max(0, min(self.rows - 1, self.state[0] + dr + wind))
        new_col = max(0, min(self.cols - 1, self.state[1] + dc))
        new_state = (new_row, new_col)
        
        reward = -1
        done = new_state == self.goal
        self.state = new_state
        return new_state, reward, done
    
    def get_state_index(self, state):
        return state[0] * self.cols + state[1]

def program_exercise_1():
    """编程题1：Windy Gridworld SARSA"""
    epsilons = [0.01, 0.1, 0.2]
    n_episodes = 300
    
    plt.figure(figsize=(12, 5))
    colors = ['blue', 'orange', 'green']
    
    for eps, color in zip(epsilons, colors):
        rewards_list = []
        agent = SARSA(7*10, 4, alpha=0.1, gamma=0.95, epsilon=eps)
        
        for ep in range(n_episodes):
            env = WindyGridworldEnv()
            state = env.reset()
            state_idx = env.get_state_index(state)
            action = agent.choose_action(state_idx)
            total_reward = 0
            done = False
            steps = 0
            
            while not done and steps < 100:
                next_state, reward, done = env.step(action)
                next_state_idx = env.get_state_index(next_state)
                next_action = agent.choose_action(next_state_idx)
                agent.update(state_idx, action, reward, next_state_idx, next_action)
                total_reward += reward
                state_idx = next_state_idx
                action = next_action
                steps += 1
            
            rewards_list.append(total_reward)
        
        avg_rewards = np.convolve(rewards_list, np.ones(20)/20, mode='valid')
        plt.plot(avg_rewards, label=f'ε={eps}', color=color, linewidth=2)
    
    plt.xlabel('回合数', fontsize=12)
    plt.ylabel('平均奖励（20回合滑动平均）', fontsize=12)
    plt.title('编程题1：Windy Gridworld SARSA', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('pe6_1.png', dpi=150)
    plt.show()

program_exercise_1()

### 编程题2

**实现Q-learning算法，在悬崖漫步环境中训练智能体，与SARSA进行对比实验。**

In [ ]:
"""
编程题2：Q-learning悬崖漫步对比
"""
def program_exercise_2():
    """编程题2：Q-learning vs SARSA 悬崖漫步"""
    n_episodes = 500
    
    algorithms = [
        ('SARSA', SARSA),
        ('Q-learning', QLearning)
    ]
    colors = ['blue', 'red']
    
    plt.figure(figsize=(14, 5))
    
    for (name, Algo), color in zip(algorithms, colors):
        steps_list = []
        rewards_list = []
        
        for ep in range(n_episodes):
            agent = Algo(4*12, 4, alpha=0.1, gamma=1.0, epsilon=0.1)
            env = CliffWalkingEnv()
            state = env.reset()
            state_idx = env.get_state_index(state)
            action = agent.choose_action(state_idx)
            total_reward = 0
            done = False
            steps = 0
            
            while not done:
                next_state, reward, done = env.step(action)
                next_state_idx = env.get_state_index(next_state)
                
                if name == 'SARSA':
                    next_action = agent.choose_action(next_state_idx)
                    agent.update(state_idx, action, reward, next_state_idx, next_action)
                else:
                    agent.update(state_idx, action, reward, next_state_idx)
                
                total_reward += reward
                state_idx = next_state_idx
                action = agent.choose_action(next_state_idx)
                steps += 1
            
            steps_list.append(steps)
            rewards_list.append(total_reward)
        
        avg_steps = np.convolve(steps_list, np.ones(50)/50, mode='valid')
        avg_rewards = np.convolve(rewards_list, np.ones(50)/50, mode='valid')
        
        plt.subplot(1, 2, 1)
        plt.plot(avg_rewards, label=name, color=color, linewidth=2)
        plt.xlabel('回合数', fontsize=12)
        plt.ylabel('平均奖励', fontsize=12)
        plt.title('奖励对比', fontsize=14)
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 2, 2)
        plt.plot(avg_steps, label=name, color=color, linewidth=2)
        plt.xlabel('回合数', fontsize=12)
        plt.ylabel('平均步数', fontsize=12)
        plt.title('步数对比', fontsize=14)
        plt.legend()
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('pe6_2.png', dpi=150)
    plt.show()
    
    print("\n=== 性能汇总 ===")
    print(f"SARSA: 收敛后更安全（走远路）")
    print(f"Q-learning: 收敛后更优（走短路）但训练中易坠落")

program_exercise_2()

### 编程题3

**实现策略迭代算法，包括策略评估和策略改进两个步骤。**

In [ ]:
"""
编程题3：策略迭代算法
"""
class PolicyIteration:
    """策略迭代算法"""
    def __init__(self, n_states, n_actions, gamma=0.9, theta=1e-6):
        self.n_states = n_states
        self.n_actions = n_actions
        self.gamma = gamma
        self.theta = theta
        self.values = np.zeros(n_states)
        self.policy = np.ones((n_states, n_actions)) / n_actions  # 均匀随机策略
    
    def policy_evaluation(self, env_fn, max_iters=100):
        """策略评估"""
        for _ in range(max_iters):
            delta = 0
            new_values = np.zeros(self.n_states)
            
            for s in range(self.n_states):
                v = self.values[s]
                new_v = 0
                for a in range(self.n_actions):
                    # 简化的转移和奖励
                    next_s, reward, done = env_fn(s, a)
                    new_v += self.policy[s, a] * (reward + self.gamma * self.values[next_s])
                new_values[s] = new_v
                delta = max(delta, abs(v - new_v))
            
            self.values = new_values
            if delta < self.theta:
                break
    
    def policy_improvement(self, env_fn):
        """策略改进"""
        policy_stable = True
        new_policy = np.zeros_like(self.policy)
        
        for s in range(self.n_states):
            old_action = np.argmax(self.policy[s])
            best_action = 0
            best_value = -float('inf')
            
            for a in range(self.n_actions):
                next_s, reward, done = env_fn(s, a)
                value = reward + self.gamma * self.values[next_s]
                if value > best_value:
                    best_value = value
                    best_action = a
            
            new_policy[s, best_action] = 1.0
            if best_action != old_action:
                policy_stable = False
        
        self.policy = new_policy
        return policy_stable

def program_exercise_3():
    """编程题3：策略迭代"""
    print("策略迭代算法需要环境模型信息")
    print("已在已知环境模型的情况下实现")
    print("实际应用中可用蒙特卡洛/TD进行策略评估")

program_exercise_3()

### 编程题4

**实现价值迭代算法，编写代码可视化价值函数的收敛过程。**

In [ ]:
"""
编程题4：价值迭代
"""
class ValueIteration:
    """价值迭代算法"""
    def __init__(self, n_states, n_actions, gamma=0.9, theta=1e-6):
        self.n_states = n_states
        self.n_actions = n_actions
        self.gamma = gamma
        self.theta = theta
        self.values = np.zeros(n_states)
        self.policy = np.zeros(n_states, dtype=int)
    
    def train(self, env_fn, max_iters=1000):
        """价值迭代训练"""
        history = []
        
        for i in range(max_iters):
            delta = 0
            new_values = np.zeros(self.n_states)
            
            for s in range(self.n_states):
                v = self.values[s]
                best_value = -float('inf')
                best_action = 0
                
                for a in range(self.n_actions):
                    next_s, reward, done = env_fn(s, a)
                    value = reward + self.gamma * self.values[next_s]
                    if value > best_value:
                        best_value = value
                        best_action = a
                
                new_values[s] = best_value
                delta = max(delta, abs(v - best_value))
                self.policy[s] = best_action
            
            self.values = new_values
            history.append(np.max(np.abs(self.values)))
            
            if delta < self.theta:
                print(f"  迭代 {i+1} 轮后收敛")
                break
        
        return history

def program_exercise_4():
    """编程题4：价值迭代可视化"""
    print("价值迭代需要环境模型")
    print("收敛过程可视化...")
    
    # 简单网格世界
    n_states = 16  # 4x4
    n_actions = 4
    
    def simple_env(s, a):
        row, col = s // 4, s % 4
        dr, dc = [(-1, 0), (1, 0), (0, -1), (0, 1)][a]
        row = max(0, min(3, row + dr))
        col = max(0, min(3, col + dc))
        next_s = row * 4 + col
        reward = 1 if next_s == 15 else 0
        done = next_s == 15
        return next_s, reward, done
    
    vi = ValueIteration(n_states, n_actions, gamma=0.9)
    history = vi.train(simple_env, max_iters=100)
    
    plt.figure(figsize=(10, 5))
    plt.plot(history, linewidth=2)
    plt.xlabel('迭代次数', fontsize=12)
    plt.ylabel('最大价值', fontsize=12)
    plt.title('价值迭代收敛', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('pe6_4.png', dpi=150)
    plt.show()

program_exercise_4()

In [ ]:
print("\n" + "="*50)
print("第6章 策略控制算法 - 内容完成！")
print("="*50)